# 02 - Prepare workforce capacity

This notebook prepares the 2025 Local policing FTE used as force capacity in the allocation model.

In [ ]:
import pandas as pd
from pathlib import Path
import sqlite3

## Settings

In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name in ["notebooks", "allocation"]:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
ALLOC_DB_PATH = DATA_DIR / "allocation_model.db"

WORKFORCE_YEAR = 2025
LOCAL_POLICING_FUNCTION = "local policing"

## Build force capacity

In [ ]:
with sqlite3.connect(ALLOC_DB_PATH) as conn:
    workforce = pd.read_sql_query("SELECT * FROM police_workforce_resources;", conn)

selected_workforce = workforce[
    (workforce["year"] == WORKFORCE_YEAR)
    & (workforce["wider_function_name"].str.lower() == LOCAL_POLICING_FUNCTION)
].copy()

selected_workforce["capacity_group"] = selected_workforce["worker_type"].apply(
    lambda worker_type: (
        "pcso_fte"
        if worker_type == "Police Community Support Officer"
        else "regular_police_staff_fte"
    )
)

force_capacity_long = (
    selected_workforce
    .groupby(["pfa_code", "pfa_name", "capacity_group"], as_index=False)
    .agg(fte=("fte", "sum"))
)

force_capacity_model = (
    force_capacity_long
    .pivot_table(
        index=["pfa_code", "pfa_name"],
        columns="capacity_group",
        values="fte",
        fill_value=0,
    )
    .reset_index()
)
force_capacity_model.columns.name = None

for col in ["pcso_fte", "regular_police_staff_fte"]:
    if col not in force_capacity_model.columns:
        force_capacity_model[col] = 0

force_capacity_model["total_capacity"] = (
    force_capacity_model["pcso_fte"]
    + force_capacity_model["regular_police_staff_fte"]
)

force_capacity_model = force_capacity_model[
    [
        "pfa_code",
        "pfa_name",
        "pcso_fte",
        "regular_police_staff_fte",
        "total_capacity",
    ]
].sort_values("pfa_name").reset_index(drop=True)

force_capacity_model

## Save capacity table

In [ ]:
with sqlite3.connect(ALLOC_DB_PATH) as conn:
    force_capacity_model.to_sql("force_capacity", conn, if_exists="replace", index=False)

print("Saved force_capacity rows:", len(force_capacity_model))
print("Total capacity:", force_capacity_model["total_capacity"].sum())